In [1]:
from dataclasses import dataclass


@dataclass
class Instruction:
    name: str
    destination: str
    sources: list


instructions = [
    Instruction("I1", "R1", ["R2", "R3"]),
    Instruction("I2", "R4", ["R5", "R6"]),
    Instruction("I3", "R7", ["R1", "R4"]),
    Instruction("I4", "R2", ["R5", "R8"]),
    Instruction("I5", "R4", ["R7", "R2"]),
    Instruction("I6", "R7", ["R8", "R9"]),
]


# BAGIAN ANGGOTA 1: RAW (Read After Write)
def has_raw_dependency(inst1, inst2):
    if inst1.destination in inst2.sources:
        return True, f"RAW (Register: {inst1.destination})"
    return False, None


# BAGIAN ANGGOTA 2: WAR (Write After Read)
def has_war_dependency(inst1, inst2):
    if inst2.destination in inst1.sources:
        return True, f"WAR (Register: {inst2.destination})"
    return False, None


# BAGIAN ANGGOTA 3: WAW (Write After Write) & Integrasi Semua Mode
def has_waw_dependency(inst1, inst2):
    if inst1.destination == inst2.destination:
        return True, f"WAW (Register: {inst1.destination})"
    return False, None


def get_dependencies(inst1, inst2, mode="ALL"):
    reasons = []
    if mode in ("RAW", "ALL"):
        ok, detail = has_raw_dependency(inst1, inst2)
        if ok:
            reasons.append(detail)
    if mode in ("WAR", "ALL"):
        ok, detail = has_war_dependency(inst1, inst2)
        if ok:
            reasons.append(detail)
    if mode in ("WAW", "ALL"):
        ok, detail = has_waw_dependency(inst1, inst2)
        if ok:
            reasons.append(detail)
    return reasons


def has_dependency(inst1, inst2, mode="ALL"):
    return len(get_dependencies(inst1, inst2, mode=mode)) > 0


def show_dependencies(instructions, mode="ALL"):
    print("\n" + "=" * 60)
    print(f"DEPENDENCY ANALYSIS (MODE: {mode})")
    print("=" * 60)
    found = False
    for i in range(len(instructions)):
        for j in range(i + 1, len(instructions)):
            for reason in get_dependencies(instructions[i], instructions[j], mode=mode):
                print(f"{instructions[j].name} depends on {instructions[i].name} [{reason}]")
                found = True
    if not found:
        print("No dependencies found.")


def sequential_execution(instructions):
    print("\n" + "=" * 60)
    print("SEQUENTIAL EXECUTION")
    print("=" * 60)
    for cycle, inst in enumerate(instructions, start=1):
        print(f"Cycle {cycle}: Executing {inst.name} ({inst.destination} = {' + '.join(inst.sources)})")
    print(f"\nTotal Sequential Cycles: {len(instructions)}")
    return len(instructions)


def parallel_execution(instructions, mode="ALL"):
    print("\n" + "=" * 60)
    print(f"INSTRUCTION-LEVEL PARALLEL EXECUTION (MODE: {mode})")
    print("=" * 60)
    completed = set()
    remaining = instructions.copy()
    cycle = 0

    while remaining:
        cycle += 1
        current_cycle = []
        for instruction in remaining:
            dependencies_satisfied = True
            for previous in instructions:
                if previous.name == instruction.name:
                    break
                if has_dependency(previous, instruction, mode=mode) and previous.name not in completed:
                    dependencies_satisfied = False
                    break
            if dependencies_satisfied:
                current_cycle.append(instruction)

        print(f"\nCycle {cycle}:")
        for instruction in current_cycle:
            print(f"  -> {instruction.name}: {instruction.destination} = {' + '.join(instruction.sources)}")
        for instruction in current_cycle:
            completed.add(instruction.name)
            remaining.remove(instruction)

    print(f"\nTotal ILP Cycles ({mode}): {cycle}")
    return cycle


def run_mode_evaluation(instructions, mode, title):
    print("\n" + "#" * 60)
    print(f"  {title}")
    print("#" * 60)
    show_dependencies(instructions, mode=mode)
    seq_cycles = len(instructions)
    ilp_cycles = parallel_execution(instructions, mode=mode)
    speedup = seq_cycles / ilp_cycles
    efficiency = (speedup / len(instructions)) * 100
    print(f"\nSequential Cycles : {seq_cycles} | ILP Cycles : {ilp_cycles}")
    print(f"Speedup           : {speedup:.2f}x | Efficiency : {efficiency:.2f}%")


def main():
    print("\n" + "*" * 60)
    print("      INSTRUCTION-LEVEL PARALLELISM SIMULATOR")
    print("*" * 60)
    sequential_execution(instructions)
    run_mode_evaluation(instructions, "RAW", "IMPLEMENTASI ANGGOTA 1: RAW (Read After Write)")
    run_mode_evaluation(instructions, "WAR", "IMPLEMENTASI ANGGOTA 2: WAR (Write After Read)")
    run_mode_evaluation(instructions, "WAW", "IMPLEMENTASI ANGGOTA 3: WAW (Write After Write)")
    run_mode_evaluation(instructions, "ALL", "INTEGRASI LENGKAP: GABUNGAN RAW + WAR + WAW")


if __name__ == "__main__":
    main()


************************************************************
      INSTRUCTION-LEVEL PARALLELISM SIMULATOR
************************************************************

SEQUENTIAL EXECUTION
Cycle 1: Executing I1 (R1 = R2 + R3)
Cycle 2: Executing I2 (R4 = R5 + R6)
Cycle 3: Executing I3 (R7 = R1 + R4)
Cycle 4: Executing I4 (R2 = R5 + R8)
Cycle 5: Executing I5 (R4 = R7 + R2)
Cycle 6: Executing I6 (R7 = R8 + R9)

Total Sequential Cycles: 6

############################################################
  IMPLEMENTASI ANGGOTA 1: RAW (Read After Write)
############################################################

DEPENDENCY ANALYSIS (MODE: RAW)
I3 depends on I1 [RAW (Register: R1)]
I3 depends on I2 [RAW (Register: R4)]
I5 depends on I3 [RAW (Register: R7)]
I5 depends on I4 [RAW (Register: R2)]

INSTRUCTION-LEVEL PARALLEL EXECUTION (MODE: RAW)

Cycle 1:
  -> I1: R1 = R2 + R3
  -> I2: R4 = R5 + R6
  -> I4: R2 = R5 + R8
  -> I6: R7 = R8 + R9

Cycle 2:
  -> I3: R7 = R1 + R4

Cycle 3:
  -> I5: R4 